In [1]:
# Data handling
import pandas as pd

# ML tools
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Model
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
# Load dataset (exported from preprocessing step)
df = pd.read_csv("../data/processed/processed_data.csv")

print(df.shape)
df.head()

(25000, 21)


,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_rating,delivery_cost,...,speed_kmph_recon,weather_mult_recon,delivery_time_hours_recon,partner_mult_recon,order_date_recon,order_ts_recon,expected_ts_recon,hour,delay_hours,delayed_flag
0,amazon logistics,automobile parts,ev bike,standard,west,clear,235.6,48.07,5,1322.21,...,30,1.0,9.332489,1.001906,21-10-2024,2024-10-21 13:00:00,2024-10-23 17:48:00,13,-43.467511,0
1,amazon logistics,clothing,bike,express,central,stormy,81.8,45.51,2,595.53,...,35,1.1,4.129935,1.001906,02-01-2024,2024-01-02 12:00:00,2024-01-02 20:00:00,12,-3.870065,0
2,amazon logistics,clothing,van,same day,north,clear,282.9,31.33,2,1608.49,...,45,1.0,7.427398,1.001906,31-05-2024,2024-05-31 11:00:00,2024-06-01 13:24:00,11,-18.972602,0
3,amazon logistics,cosmetics,ev bike,two day,central,hot,88.6,8.67,3,469.01,...,30,1.1,3.997011,1.001906,03-01-2024,2024-01-03 17:00:00,2024-01-05 17:00:00,17,-44.002989,0
4,amazon logistics,cosmetics,ev van,two day,east,rainy,204.2,8.09,4,1045.27,...,40,1.2,6.933351,1.001906,19-03-2024,2024-03-19 13:00:00,2024-03-21 17:48:00,13,-45.866649,0


In [4]:
# Target variable
y = df["delayed_flag"]

# Features
X = df.drop(columns=["delayed_flag", "delay_hours"])

In [5]:
categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(exclude="object").columns

print(categorical_cols)
print(numerical_cols)

Index(['delivery_partner', 'package_type', 'vehicle_type', 'delivery_mode',
       'region', 'weather_condition', 'order_date_recon', 'order_ts_recon',
       'expected_ts_recon'],
      dtype='object')
Index(['distance_km', 'package_weight_kg', 'delivery_rating', 'delivery_cost',
       'expected_time_hours_recon', 'speed_kmph_recon', 'weather_mult_recon',
       'delivery_time_hours_recon', 'partner_mult_recon', 'hour'],
      dtype='object')


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)

In [8]:
model = RandomForestClassifier(random_state=42)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", model)
])

In [9]:
pipeline.fit(X_train, y_train)
print("Training completed")

Training completed


In [10]:
y_pred = pipeline.predict(X_test)

In [11]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9976

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      4853
           1       1.00      0.92      0.96       147

    accuracy                           1.00      5000
   macro avg       1.00      0.96      0.98      5000
weighted avg       1.00      1.00      1.00      5000


Confusion Matrix:
 [[4853    0]
 [  12  135]]
